In [33]:
import pandas as pd
import os
import requests
from functools import reduce # tool for merging multiple dataframes together easily
from tqdm import tqdm # library for showing progress bars

# Create output directory if it doesn't exist
os.makedirs("data/raw", exist_ok=True)

In [34]:
def get_asylum_data(cities, decision, represented, col_name, city_label=False, nationwide=False):
    cases_dfs = []
    if nationwide:
        cities = {'Nationwide': 'All'}
    for city in tqdm(cities.values()):
        url = f'https://tracreports.org/phptools/immigration/asylum/graph.php?stat=count&timescale=fymon&base_city={str(city)}&represented={str(represented)}&denial2={str(decision)}&timeunit=number'
        response = requests.get(url)
        data = response.json()
        df = pd.DataFrame(data['timeline'])
        if city_label:
            df['city'] = [k for k, v in cities.items() if v == city][0]
        df.rename(columns={'number': col_name}, inplace=True)
        df.drop(columns='percent', inplace=True) # drop pct column which we will recalculate
        cases_dfs.append(df)
    cases = pd.concat(cases_dfs)
    return cases

In [35]:
# Create cities dict. Each city has a numeric code in the API. Philly's is 30 (base_city = 30 parameter below)
# To get the data for the whole country, base_city=All is used, reflected in the function above
cities = {'Philadelphia': 30}

In [36]:
## PHILADELPHIA

## These two will be combined into one dataframe on overall cases
# Get all cases. Ensure "All" is passed with a captial A for representation and decision parameters
all_cases = get_asylum_data(cities, decision='All', col_name='all_cases', represented='All', city_label=True)
# Get denied cases. The code for Denied is 1
denied_cases = get_asylum_data(cities, decision=1, col_name='denied_cases', represented='All')

## These four will be combined into one dataframe on asylum outcomes by legal representation. 
# Only one of them needs a "city label" because they will later be combined
# Get all cases with representation. The code for represented is 2
all_represented = get_asylum_data(cities, decision='All', col_name='all_represented', represented=2, city_label=True)
# Get all cases without representation. The code for not represented is 1
all_not_represented = get_asylum_data(cities, decision='All', col_name='all_not_represented', represented=1)
# Get all cases with representation that were denied
represented_denied = get_asylum_data(cities, decision=1, col_name='represented_denied', represented=2)
# Get all cases without representation that were denied
not_represented_denied = get_asylum_data(cities, decision=1, col_name='not_represented_denied', represented=1)

100%|██████████| 1/1 [00:00<00:00,  6.64it/s]


In [37]:
## NATIONWIDE - repeating the above set of function calls for the whole country.

all_cases_us = get_asylum_data(cities, decision='All', col_name='all_cases', represented='All', nationwide=True, city_label=True)
denied_cases_us = get_asylum_data(cities, decision=1, col_name='denied_cases', represented='All', nationwide=True)
all_represented_us = get_asylum_data(cities, decision='All', col_name='all_represented', represented=2, nationwide=True, city_label=True)
all_not_represented_us = get_asylum_data(cities, decision='All', col_name='all_not_represented', represented=1, nationwide=True)
represented_denied_us = get_asylum_data(cities, decision=1, col_name='represented_denied', represented=2, nationwide=True)
not_represented_denied_us = get_asylum_data(cities, decision=1, col_name='not_represented_denied', represented=1, nationwide=True)

100%|██████████| 1/1 [00:00<00:00,  6.60it/s]


In [38]:
# Merge all the dataframes together

# Make lists of dfs
dfs_philly  = [all_cases, denied_cases]
dfs_rep_philly = [all_represented, all_not_represented, represented_denied, not_represented_denied]
dfs_nationwide = [all_cases_us, denied_cases_us]
dfs_rep_nationwide = [all_represented_us, all_not_represented_us, represented_denied_us, not_represented_denied_us]

# Merge philly and us dataframes together on month column
asylum_data_philly = reduce(lambda left, right: pd.merge(left, right, on="fymon"), dfs_philly)
asylum_data_philly_rep = reduce(lambda left, right: pd.merge(left, right, on="fymon"), dfs_rep_philly)
asylum_data_us = reduce(lambda left, right: pd.merge(left, right, on="fymon"), dfs_nationwide)
asylum_data_us_rep = reduce(lambda left, right: pd.merge(left, right, on="fymon"), dfs_rep_nationwide)

# Concatenate philly and us dataframes together because they have the same column names
asylum_data = pd.concat([asylum_data_philly, asylum_data_us], axis=0)
asylum_data_rep = pd.concat([asylum_data_philly_rep, asylum_data_us_rep], axis=0)

In [39]:
asylum_data

,fymon,all_cases,city,denied_cases
0,2000-10,68,Philadelphia,32
1,2000-11,46,Philadelphia,27
2,2000-12,56,Philadelphia,31
3,2001-01,59,Philadelphia,40
4,2001-02,42,Philadelphia,26
...,...,...,...,...
298,2025-08,10371,Nationwide,7881
299,2025-09,10602,Nationwide,8463
300,2025-10,10616,Nationwide,8615
301,2025-11,7940,Nationwide,6558


In [40]:
asylum_data_rep

,fymon,all_represented,city,all_not_represented,represented_denied,not_represented_denied
0,2000-10,48,Philadelphia,20,15,17
1,2000-11,24,Philadelphia,22,12,15
2,2000-12,29,Philadelphia,27,14,17
3,2001-01,37,Philadelphia,22,25,15
4,2001-02,31,Philadelphia,11,18,8
...,...,...,...,...,...,...
298,2025-08,8354,Nationwide,2017,6248,1633
299,2025-09,8570,Nationwide,2032,6804,1659
300,2025-10,8440,Nationwide,2176,6738,1877
301,2025-11,6373,Nationwide,1567,5184,1374


In [41]:
asylum_data.to_csv("data/raw/asylum_cases_philadelphia_us.csv", index=False)
asylum_data_rep.to_csv("data/raw/asylum_cases_representation_philadelphia_us.csv", index=False)